# Training Helpers Validation Tutorial

This notebook checks the optional supervised training helpers:

$$
\text{model},\ \text{loader},\ \text{loss},\ \text{optimizer}
\quad\longrightarrow\quad
\text{history},\ \text{metric},\ \text{checkpoint}.
$$

The helpers do not define a SILVA architecture by themselves. They provide the
repeatable training loop around any PyTorch model, including models built from
`silva_networks`.

<!-- silva-numbered-citations:start -->
**Numbered literature:** [1](https://jseluis.github.io/silva-networks/paper/references/#ref-1), [4](https://jseluis.github.io/silva-networks/paper/references/#ref-4), [39](https://jseluis.github.io/silva-networks/paper/references/#ref-39). Each number opens the complete citation and its primary external source.
<!-- silva-numbered-citations:end -->


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [
        Path.cwd(),
        Path("/content/silva-networks"),
        Path("/content/drive/MyDrive/silva-networks"),
    ]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif IN_COLAB and importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [ ]:
from pathlib import Path
import tempfile

import torch
from torch.utils.data import DataLoader, TensorDataset

from silva_networks import (
    TrainConfig,
    evaluate,
    fit_supervised,
    resolve_device,
    seed_everything,
)

device = resolve_device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(10)
device

## Synthetic Classification Data

Use a tiny linearly separable dataset. The batch contract here is the ordinary
PyTorch `(x, y)` tuple:

$$
x\in\mathbb R^{N\times d},
\qquad
y\in\{0,1\}^N.
$$

In [ ]:
x_pos = torch.randn(24, 3) + torch.tensor([1.5, 0.0, 0.0])
x_neg = torch.randn(24, 3) + torch.tensor([-1.5, 0.0, 0.0])
x = torch.cat([x_pos, x_neg], dim=0)
y = torch.cat([torch.ones(24, dtype=torch.long), torch.zeros(24, dtype=torch.long)])

loader = DataLoader(TensorDataset(x, y), batch_size=12, shuffle=True)
val_loader = DataLoader(TensorDataset(x, y), batch_size=16)
model = torch.nn.Sequential(
    torch.nn.Linear(3, 12),
    torch.nn.Tanh(),
    torch.nn.Linear(12, 2),
)

## Fit and Evaluate

For classification, `loss="auto"` becomes cross entropy and `metric="auto"`
becomes accuracy. The training helper moves the model and batches to the
requested device.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    checkpoint = Path(tmp) / "training.pt"
    result = fit_supervised(
        model,
        loader,
        val_loader,
        config=TrainConfig(
            task="classification",
            epochs=3,
            lr=0.05,
            optimizer="adam",
            gradient_clipping=1.0,
            device=device,
            seed=10,
            checkpoint_path=checkpoint,
        ),
    )
    evaluation = evaluate(model, val_loader, device=device)
    print("epochs:", len(result.history))
    print("best epoch:", result.best_epoch)
    print("metric:", evaluation.metric_name, round(evaluation.metric, 4))
    print("checkpoint exists:", checkpoint.exists())

## Resume

When `resume=True`, the helper reloads the model, optimizer, scheduler if
present, and history from the checkpoint path.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    checkpoint = Path(tmp) / "resume.pt"
    _ = fit_supervised(
        model,
        loader,
        config=TrainConfig(epochs=1, lr=0.01, device=device, checkpoint_path=checkpoint),
    )
    resumed = fit_supervised(
        model,
        loader,
        config=TrainConfig(epochs=2, lr=0.01, device=device, checkpoint_path=checkpoint, resume=True),
    )
    print("history after resume:", len(resumed.history))

## Citation

If this notebook or package is used, cite:

```text
Dr. Jose Luis Silva. SILVA Networks. Version 1.0.0. MIT License.
https://github.com/jseluis/silva-networks
https://doi.org/10.5281/zenodo.21770099
```

When training SILVA models in connection with the SILVA Networks paper, cite
the paper as well.

## Where to Go Next

| Question | Page |
| --- | --- |
| Which training objects and result fields are public? | [Training API](https://jseluis.github.io/silva-networks/api/training/) |
| What evidence should a trained experiment report? | [Reconstructing Paper Experiments](https://jseluis.github.io/silva-networks/learn/reconstructing-paper-experiments/) |
| Which measured outputs are currently published? | [Results](https://jseluis.github.io/silva-networks/results/) |
